# Topic prevalence over time

### Overview

Does the share of reviews assigned to each topic change across the corpus period?


**Input:** `data/stylecom_cleaned_annotated.csv` (the per-review LLM topic labels).
**Output:** none saved; the trend table is read inline.

**Pipeline:**
1. Setup: imports, parameters, load, and derive the balanced subset.
2. Trend model: per-topic logistic slope on year, season fixed effect, SEs clustered
   by designer, BH-corrected across the ten topics.
3. Non-monotonic (quadratic-year) check for rise-then-fall shapes a linear slope would
   miss.

## 1. Setup

In [33]:
# Cell 1: Imports

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from statsmodels.stats.multitest import multipletests

In [34]:
# Cell 2: Parameters

# "all" = full corpus; "balanced" = houses with >= MIN_REVIEWS
SUBSET = "balanced"        
MIN_REVIEWS = 24

N_TOPICS = 10
ANNOTATED_CSV = "data/stylecom_cleaned_annotated.csv"

In [42]:
# Cell 3: Load and prepare

annotated = pd.read_csv(ANNOTATED_CSV)

_MISSING = ["", "nan", "none"]
_auth = annotated["author"].astype("string").str.strip().str.lower()
annotated = annotated[~(_auth.isna() | _auth.isin(_MISSING))].reset_index(drop=True)
annotated["year"] = pd.to_datetime(annotated["date"], format="%d-%b-%y").dt.year
annotated["year_c"] = annotated["year"] - 2000

brand_counts = annotated["designer"].value_counts()
SUBSET_DESIGNERS = sorted(brand_counts[brand_counts >= MIN_REVIEWS].index)

print(f"{len(annotated)} reviews, {annotated['designer'].nunique()} designers")
print(f"subset (>= {MIN_REVIEWS} reviews): {len(SUBSET_DESIGNERS)} designers, "
      f"{annotated['designer'].isin(SUBSET_DESIGNERS).sum()} reviews")

6492 reviews, 709 designers
subset (>= 24 reviews): 65 designers, 1897 reviews


In [44]:
# Cell 4: Select the subset

if SUBSET == "all":
    data = annotated.copy()
    label = "full corpus"
elif SUBSET == "balanced":
    if not SUBSET_DESIGNERS:
        raise ValueError('SUBSET = "balanced" but no brand clears MIN_REVIEWS - lower it')
    data = annotated[annotated["designer"].isin(SUBSET_DESIGNERS)].copy()
    label = f"balanced subset ({len(SUBSET_DESIGNERS)} designers, >= {MIN_REVIEWS} reviews)"
else:
    raise ValueError(f'SUBSET must be "all" or "balanced", got {SUBSET!r}')


print(f"{len(data)} reviews ({len(data) / len(annotated):.1%} of corpus), "
      f"{data['designer'].nunique()} designers")

1897 reviews (29.2% of corpus), 65 designers


## 2. Trend model: per-topic logistic slope on year

In [37]:
# Cell 5: Trend specification

specs = [
    ("base", "y ~ year_c + C(season)", data),
]

for name, _, d in specs:
    print(f"  {name:12} {len(d):5} reviews")

  base          1897 reviews


In [38]:
# Cell 6: Per-topic logistic slope on year, BH-corrected

rows = []
for spec_name, formula, spec_data in specs:
    for k in range(N_TOPICS):
        d = spec_data.copy()
        d["y"] = (d["topic_number"] == k).astype(int)
        try:
            m = smf.logit(formula, data=d).fit(
                disp=0, cov_type="cluster", cov_kwds={"groups": d["designer"]}
            )
            rows.append({
                "spec": spec_name,
                "topic": k,
                "coef": m.params["year_c"],
                "odds_ratio_per_year": np.exp(m.params["year_c"]),
                "p": m.pvalues["year_c"],
            })
        except Exception as e:
            print(f"  ! {spec_name} topic {k} failed: {type(e).__name__}")

trends = pd.DataFrame(rows)

# correct across the 10 topics within each spec
trends["p_adj"] = np.nan
for spec_name in trends["spec"].unique():
    rows_i = trends["spec"] == spec_name
    trends.loc[rows_i, "p_adj"] = multipletests(trends.loc[rows_i, "p"], method="fdr_bh")[1]

trends["sig"] = trends["p_adj"] < 0.05
trends["direction"] = np.where(trends["coef"] > 0, "up", "down")
trends.round(4)

,spec,topic,coef,odds_ratio_per_year,p,p_adj,sig,direction
0,base,0,0.0248,1.0251,0.2398,0.4795,False,up
1,base,1,0.0076,1.0076,0.6365,0.8037,False,up
2,base,2,0.0003,1.0003,0.9889,0.9929,False,up
3,base,3,0.0002,1.0002,0.9929,0.9929,False,up
4,base,4,0.0764,1.0794,0.0188,0.0938,False,up
5,base,5,0.0129,1.0130,0.6430,0.8037,False,up
6,base,6,0.0175,1.0177,0.3465,0.5776,False,up
7,base,7,-0.0428,0.9581,0.1269,0.3173,False,down
8,base,8,-0.0569,0.9447,0.0478,0.1594,False,down
9,base,9,-0.0391,0.9616,0.0056,0.0562,False,down


In [41]:
# Cell 7: Base trend table, most significant first

summary = (trends.set_index("topic")[["odds_ratio_per_year", "p", "p_adj", "direction"]]
           .round(3)
           .sort_values("p_adj"))
summary

,odds_ratio_per_year,p,p_adj,direction
topic,,,,
9,0.962,0.006,0.056,down
4,1.079,0.019,0.094,up
8,0.945,0.048,0.159,down
7,0.958,0.127,0.317,down
0,1.025,0.240,0.480,up
6,1.018,0.347,0.578,up
1,1.008,0.636,0.804,up
5,1.013,0.643,0.804,up
2,1.000,0.989,0.993,up


## 3. Non-monotonic (quadratic-year) check

In [ ]:
# Cell 8: Non-monotonic (quadratic-year) check

for k in range(N_TOPICS):
    d = data.copy()
    d["y"] = (d["topic_number"] == k).astype(int)
    m = smf.logit("y ~ year_c + I(year_c**2) + C(season)", data=d).fit(
        disp=0, cov_type="cluster", cov_kwds={"groups": d["designer"]}
    )
    p_sq = m.pvalues["I(year_c ** 2)"]
    flag = "  <- curved" if p_sq < 0.05 else ""
    print(f"topic {k}: p(year^2) = {p_sq:.4f}{flag}")

topic 0: p(year^2) = 0.3728
topic 1: p(year^2) = 0.1199
topic 2: p(year^2) = 0.6553
topic 3: p(year^2) = 0.1661
topic 4: p(year^2) = 0.0069  <- curved
topic 5: p(year^2) = 0.1114
topic 6: p(year^2) = 0.3354
topic 7: p(year^2) = 0.1914
topic 8: p(year^2) = 0.8475
topic 9: p(year^2) = 0.0086  <- curved
